In [ ]:
import requests
import time
import json
import re
import pandas as pd
from bs4 import BeautifulSoup
import spacy

In [ ]:
nlp = spacy.load("de_core_news_md")
CLIMATE_KEYWORDS = ["klima", "klimawandel", "klimakrise", "erderwärmung", "globale erwärmung", "treibhauseffekt", "treibhausgas",
    "co2", "kohlendioxid", "emission", "emissionen", "klimaschutz", "klimapolitik", "klimaneutral", "dekarbonisierung",
    
    "klimaziel", "klimaziele", "co2-preis", "emissionshandel", "klimaschutzgesetz", "paris-abkommen", "pariser abkommen",
    "eu-klimapaket", "green deal", "klimaplan", "klimabericht", "ipcc",

    "energiewende", "erneuerbare energien", "windkraft", "solarenergie", "photovoltaik", "wasserstoff",
    "kohlekraft", "kohleausstieg", "atomkraft", "stromnetz", "netzausbau", "e-mobilität", "verbrennerverbot", "industrieemissionen",
    
    "hitzewelle", "dürre", "hochwasser", "überschwemmung", "starkregen", "waldbrand", "extremwetter",
    "gletscherschmelze", "meeresspiegel", "wasserknappheit", "klimafolgen", "artensterben",

    "klimarisiko", "nachhaltigkeit", "nachhaltige investitionen", "esg", "grüne finanzierung", "klimafonds",
    "energiepreise", "strompreis", "gaspreis",

    "landwirtschaft", "trockenheit", "ernteschäden", "umweltschutz", "biodiversität", "naturschutz", "ökosystem", "umweltpolitik",

    "fridays for future", "klimaaktivisten", "klimaprotest", "letzte generation", "klimabewegung", "klimadebatte"]

STRICT_KEYWORDS = ["klima", "klimakrise", "klimawandel", "erderwärmung", "co2", "kohlendioxid", "emission", "emissionen",
    "energiewende", "klimaschutz", "klimaneutral", "treibhauseffekt", "treibhausgas", "hitzewelle", "dürre", "hochwasser",
    "wasserknappheit", "starkregen", "waldbrand"]
TIME_WINDOW = "y6"
OFFSET_STEP = 10
OUTPUT_CSV = "welt_scrapped.csv"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Accept": "application/json",
    "Referer": "https://www.welt.de/suche",}
SOURCE = "WELT"
LANGUAGE = "de"
CLASSIFICATION_KEYWORDS = {
    "Climate_Policy": ["gesetz", "politik", "regierung", "beschluss", "verordnung","ziel", "klimaziel", "bundestag", "eu", "parlament", "ministerium"],
    "Climate_Science": ["studie", "forschung", "wissenschaft", "ipcc", "daten", "analyse", "bericht", "modell", "forscher"],
    "Energy_Transition": ["energiewende", "erneuerbar", "solar", "windkraft", "kohlekraft", "atomkraft", "wasserstoff", "stromnetz"],
    "Climate_Economy": ["kosten", "industrie", "wirtschaft", "markt", "unternehmen", "investition", "preis", "arbeitsplätze"],
    "Climate_Activism": ["protest", "demonstration", "aktivisten", "fridays for future", "ngo", "bewegung"],
    "Climate_Impact": ["hitzewelle", "dürre", "hochwasser", "überschwemmung", "starkregen", "waldbrand", "extremwetter"],
    "Climate_Geopolitics": ["china", "usa", "eu", "russland", "international", "global", "weltweit", "g7", "g20"],
    "Climate_Opinion": ["meinung", "kommentar", "kolumne", "leitartikel", "debatte", "gastbeitrag"]}

In [ ]:
def clean_text(text):
    if not text:
        return None
    return re.sub(r"\s+", " ", text).strip()
def strict_relevance(title, content):
    title = title.lower()
    content_l = content.lower()
    lead = " ".join(content_l.split()[:300])
    title_hit = any(k in title for k in STRICT_KEYWORDS)
    lead_hits = sum(k in lead for k in STRICT_KEYWORDS)
    return title_hit or lead_hits >= 3
def extract_jsonld(soup):
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string)
        except:
            continue
        if isinstance(data, dict) and data.get("@type") == "NewsArticle":
            content = clean_text(data.get("articleBody"))
            headline = clean_text(data.get("headline"))
            author = None
            auth = data.get("author")
            if isinstance(auth, dict):
                author = auth.get("name")
            elif isinstance(auth, list) and auth:
                author = auth[0].get("name")
            return content, headline, clean_text(author)
    return None, None, None
def extract_fallback_content(soup):
    texts = []
    for p in soup.find_all("p"):
        t = p.get_text().strip()
        if len(t) < 80:
            continue
        if any(x in t.lower() for x in ["anzeige", "newsletter", "datenschutz", "jetzt abonnieren", "quelle:", "bild:"]):
            continue
        texts.append(t)
    content = clean_text(" ".join(texts))
    return content if content and len(content) > 800 else None
def analyze_text(text):
    doc = nlp(text)
    actors = sorted(set(ent.text for ent in doc.ents if ent.label_ in ["PER", "ORG"]))
    sentences = list(doc.sents)
    return actors, len(actors), len(sentences), len(text)
def classify_article(title, intro, content):
    text = f"{title} {intro} {' '.join(content.split()[:300])}".lower()
    scores = {}
    for cat, keywords in CLASSIFICATION_KEYWORDS.items():
        scores[cat] = sum(k in text for k in keywords)
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "Other_Climate"

In [ ]:
rows = []
seen_urls = set()

In [ ]:
for keyword in CLIMATE_KEYWORDS:
    print(f"\nKeyword: {keyword}")
    offset = 0
    while True:
        api_url = f"https://www.welt.de/api/search/{keyword}"
        params = {"offset": offset, "restrictBy": TIME_WINDOW}
        try:
            r = requests.get(api_url, params=params, headers=HEADERS, timeout=15)
            data = r.json()
        except:
            break
        items = data.get("items", [])
        if not items:
            break
        for item in items:
            url = item.get("url")
            if not url or url in seen_urls:
                continue
            seen_urls.add(url)
            if item.get("premium"):
                continue
            try:
                html = requests.get(url, headers=HEADERS, timeout=15).text
            except:
                continue
            soup = BeautifulSoup(html, "html.parser")
            content, headline, author = extract_jsonld(soup)
            if not content:
                content = extract_fallback_content(soup)
            if not headline:
                h1 = soup.find("h1")
                headline = clean_text(h1.get_text()) if h1 else None
            if not content or not headline:
                continue
            if not strict_relevance(headline, content):
                continue
            actors, actor_count, sent_count, length = analyze_text(content)
            article_class = classify_article(headline, item.get("intro"), content)
            rows.append({
                "URL": url,
                "Source": SOURCE,
                "Language": LANGUAGE,
                "Published_Date": item.get("publicationDate"),
                "Keyword_Matched": keyword,
                "Article_Classification": article_class,
                "Headline": headline,
                "Intro": clean_text(item.get("intro")),
                "Content": content,
                "Content_Length": length,
                "Sentence_Count": sent_count,
                "Actors": ", ".join(actors),
                "Actor_Count": actor_count,
                "Author": author
            })

            print("Saved")
            time.sleep(1)
        offset += OFFSET_STEP
        time.sleep(0.5)

In [ ]:
df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"File saved as: {OUTPUT_CSV}")
print(f"Total number of articles scrapped in Welt: {len(df)}")